In [1]:
import pandas as pd
import joblib as jb

In [2]:
url="../data/Telco-Customer-Churn.csv"
df= pd.read_csv(url)

In [3]:
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [12]:
def prepare_data(df: pd.DataFrame):
    df = df.copy()

    df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
    
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce')
    
    X = df.drop(columns=['customerID', 'Churn'])
    y = df['Churn']
    
    return X, y

In [13]:
def build_preprocessor(X: pd.DataFrame):
    num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    cat_cols = X.select_dtypes(include=['str']).columns.tolist()  

    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    cat_pipeline = Pipeline([
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])

    return ColumnTransformer(transformers=[
        ('nums', num_pipeline, num_cols),
        ('cat', cat_pipeline, cat_cols)
    ])

In [14]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

In [15]:
X, y = prepare_data(df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

preprocessor = build_preprocessor(X)

pipelines = {
    'Decision Tree': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', DecisionTreeClassifier(max_depth=6, random_state=42))
    ]),
    'Linear SVC': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', SVC(kernel='linear', probability=True, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(random_state=42))
    ])
}

In [16]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,roc_auc_score

In [17]:
results = []

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    probs = pipe.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)

    print(f"=== {name} ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"ROC-AUC  : {auc:.4f}\n")

    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall (Churn)': rec,
        'ROC-AUC': auc,
        'pipeline_object': pipe
    })


=== Decision Tree ===
Accuracy : 0.7913
Precision: 0.6342
Recall   : 0.5053
ROC-AUC  : 0.8351



/home/sayak/Documents/RetailMind AI/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


=== Linear SVC ===
Accuracy : 0.7878
Precision: 0.6176
Recall   : 0.5267
ROC-AUC  : 0.8252

=== Random Forest ===
Accuracy : 0.7779
Precision: 0.6034
Recall   : 0.4759
ROC-AUC  : 0.8162



In [19]:
results_df = pd.DataFrame(results).sort_values(by=['Recall (Churn)', 'ROC-AUC'], ascending=False).reset_index(drop=True)

print("=== Final Ranking by Recall ===")
print(results_df.drop(columns=['pipeline_object']).to_string(index=False))

# Select and save champion model
champion_row = results_df.iloc[0]
champion_name = champion_row['Model']
champion_pipeline = champion_row['pipeline_object']

champion_path = '../models/champion_churn_pipeline.joblib'
jb.dump(champion_pipeline, champion_path)

print(f"\nChampion Selected: {champion_name} (Recall: {champion_row['Recall (Churn)']:.4f})")
print(f"Persisted model saved to {champion_path}")

=== Final Ranking by Recall ===
        Model  Accuracy  Precision  Recall (Churn)  ROC-AUC
   Linear SVC  0.787793   0.617555        0.526738 0.825232
Decision Tree  0.791341   0.634228        0.505348 0.835093
Random Forest  0.777857   0.603390        0.475936 0.816159

Champion Selected: Linear SVC (Recall: 0.5267)
Persisted model saved to ../models/champion_churn_pipeline.joblib
